# PYNQ-Z1 TinyUSB UART / STM32 Screen Test

Default physical link: PYNQ PS UART0 (MIO14/MIO15, /dev/ttyPS0) -> STM32 UART1.

This notebook uses **115200, 8 data bits, even parity, 1 stop bit (8E1)**. The STM32 firmware must use the same setting. The existing CR/LF text protocol stays unchanged: STM32 sends WCFG; PYNQ returns STATUS, TEST, LOG, or ERR.


In [ ]:
# Cell 1: load the existing MAX5885 overlay and discover the TinyUSB CDC UART.
from time import sleep, time
import glob
import os
import select
import termios
try:
    import serial
    from serial.tools import list_ports
except ModuleNotFoundError:
    serial = None
from pynq import Overlay, MMIO

BITFILE = 'base_add.bit'
UART_PORT = '/dev/ttyPS0'  # Board PROG/UART: PS UART0 on MIO14/MIO15. Use None only for an external TinyUSB CDC device.
UART_BAUD = 115200
UART_PARITY = serial.PARITY_EVEN if serial is not None else 'E'
UART_TIMEOUT_S = 0.20
APPLY_WCFG_TO_DAC = False  # Keep False for the first link test; set True after TX/RX passes.

overlay = Overlay(BITFILE)
MAX5885_BASE = 0x40001000
max5885_ip = MMIO(MAX5885_BASE, 0x1000)
print('Overlay:', BITFILE)
print('MAX5885 version=0x%08X, sample_rate=%d' % (max5885_ip.read(0x38), max5885_ip.read(0x3C)))

class TermiosSerial:
    # Standard-library fallback for Linux CDC/USB serial, configured as 115200 8E1.
    def __init__(self, port, baudrate, timeout=0.2, write_timeout=1.0):
        if baudrate != 115200:
            raise ValueError('Fallback currently supports the fixed 115200 baud protocol only.')
        self.port, self.baudrate, self.parity = port, baudrate, 'E'
        self.timeout, self.write_timeout = timeout, write_timeout
        self.fd = os.open(port, os.O_RDWR | os.O_NOCTTY | os.O_NONBLOCK)
        attrs = termios.tcgetattr(self.fd)
        attrs[0] = termios.INPCK | termios.IGNPAR  # Check then discard bad-parity bytes.
        attrs[1] = 0
        attrs[2] = termios.CLOCAL | termios.CREAD | termios.CS8 | termios.PARENB
        attrs[2] &= ~termios.PARODD
        if hasattr(termios, 'CRTSCTS'):
            attrs[2] &= ~termios.CRTSCTS
        attrs[3] = 0
        attrs[4] = termios.B115200
        attrs[5] = termios.B115200
        attrs[6][termios.VMIN] = 0
        attrs[6][termios.VTIME] = 0
        termios.tcsetattr(self.fd, termios.TCSANOW, attrs)
    def read(self, size=1):
        ready, _, _ = select.select([self.fd], [], [], self.timeout)
        return os.read(self.fd, size) if ready else b''
    def write(self, data):
        deadline, sent = time() + self.write_timeout, 0
        while sent < len(data):
            if time() >= deadline:
                raise TimeoutError('UART write timed out.')
            try:
                sent += os.write(self.fd, data[sent:])
            except BlockingIOError:
                select.select([], [self.fd], [], 0.02)
        return sent
    def flush(self):
        termios.tcdrain(self.fd)
    def reset_input_buffer(self):
        termios.tcflush(self.fd, termios.TCIFLUSH)
    def reset_output_buffer(self):
        termios.tcflush(self.fd, termios.TCOFLUSH)

ports = list(list_ports.comports()) if serial is not None else []
for port in ports:
    print('UART candidate:', port.device, '|', port.description, '|', port.hwid)
if serial is None:
    print('pyserial is absent; using Linux termios fallback (no network installation needed).')

if UART_PORT is None:
    preferred = [p.device for p in ports if any(k in (p.description + ' ' + p.hwid).lower()
                 for k in ('tinyusb', 'cdc', 'usb serial', 'usb vid:pid'))]
    fallback = sorted(glob.glob('/dev/ttyACM*') + glob.glob('/dev/ttyUSB*'))
    candidates = preferred + [p for p in fallback if p not in preferred]
    if not candidates:
        raise RuntimeError('No external TinyUSB CDC UART found. Connect it to the PYNQ USB HOST port, or set UART_PORT to /dev/ttyPS0 for the board PS UART.')
    UART_PORT = candidates[0]
elif not os.path.exists(UART_PORT):
    raise RuntimeError('UART device does not exist: %s. Run !ls -l /dev/ttyPS* /dev/ttyACM* /dev/ttyUSB*' % UART_PORT)
elif UART_PORT == '/dev/ttyPS0':
    print('Using board PS UART0 on MIO14/MIO15. Jupyter must be reached through Ethernet/Wi-Fi, not this serial console.')

if serial is not None:
    uart = serial.Serial(
        port=UART_PORT, baudrate=UART_BAUD, bytesize=serial.EIGHTBITS,
        parity=UART_PARITY, stopbits=serial.STOPBITS_ONE,
        timeout=UART_TIMEOUT_S, write_timeout=1.0,
    )
else:
    uart = TermiosSerial(UART_PORT, UART_BAUD, UART_TIMEOUT_S, 1.0)
uart.reset_input_buffer()
uart.reset_output_buffer()
print('Opened %s: %d 8E1; parity=%s' % (uart.port, uart.baudrate, uart.parity))


In [ ]:
# Cell 2: text transport helpers. Every application message is one ASCII CR/LF line.
def uart_send(line):
    line = str(line).strip('\r\n')
    if not line or any(ord(ch) < 0x20 or ord(ch) > 0x7E for ch in line):
        raise ValueError('UART protocol accepts non-empty printable ASCII only.')
    uart.write((line + '\r\n').encode('ascii'))
    uart.flush()
    print('TX:', line)

def uart_read_line(timeout_s=1.0):
    deadline = time() + timeout_s
    raw = bytearray()
    while time() < deadline:
        ch = uart.read(1)
        if not ch:
            continue
        if ch == b'\n':
            line = raw.rstrip(b'\r').decode('ascii', errors='replace')
            print('RX:', line)
            return line
        if len(raw) >= 255:
            raise RuntimeError('Received UART line exceeds STM32 protocol limit (255 bytes).')
        raw.extend(ch)
    return None

def uart_drain(duration_s=0.15):
    lines = []
    deadline = time() + duration_s
    while time() < deadline:
        line = uart_read_line(timeout_s=min(0.05, deadline - time()))
        if line is not None:
            lines.append(line)
    return lines


In [ ]:
# Cell 3: UART 8E1 link test. The STM32 screen should show these messages.
uart_drain()
uart_send('STATUS,PYNQ UART 8E1 READY')
uart_send('TEST,UART TX OK 8E1')
uart_send('LOG,TinyUSB UART link active')

print('Change one setting on the STM32 screen or reboot it. Waiting 10 s for its WCFG line...')
received = []
deadline = time() + 10.0
while time() < deadline:
    line = uart_read_line(timeout_s=0.25)
    if line is not None:
        received.append(line)
        if line.upper().startswith('WCFG,'):
            uart_send('TEST,UART RX OK 8E1')
            break

if not any(line.upper().startswith('WCFG,') for line in received):
    print('No WCFG received. Check TinyUSB TX/RX crossover, common GND, and STM32 115200 8E1 configuration.')


In [ ]:
# Cell 4: decode the WCFG format emitted by the STM32 screen.
# With APPLY_WCFG_TO_DAC=False this validates and echoes only.
# Set it True after Cell 3 passes, then run Cell 5 while changing STM32 UI settings.
from pynqz1_diansai2_max5885 import MAX5885Signal

VALID_OUTPUTS = {'SD', 'SM', 'SOUT', 'DC', 'SQUARE', 'MOD_SINE'}

def parse_wcfg(line):
    fields = [part.strip() for part in line.split(',')]
    if len(fields) != 13 or fields[0].upper() != 'WCFG':
        raise ValueError('Expected WCFG,fc_hz,mode,sd_mv,sd_phase,am_depth,sm_delay,sm_phase,sm_atten,out_a,out_b,mod_hz,square_hz')
    cfg = {
        'carrier_hz': int(fields[1]), 'mode': fields[2].upper(), 'sd_vrms': int(fields[3]) / 1000.0,
        'sd_phase_deg': int(fields[4]), 'am_depth_percent': int(fields[5]),
        'sm_delay_ns': int(fields[6]), 'sm_phase_deg': int(fields[7]), 'sm_atten_db': int(fields[8]),
        'out_a': fields[9].upper(), 'out_b': fields[10].upper(),
        'mod_hz': int(fields[11]), 'square_hz': int(fields[12]),
    }
    if cfg['mode'] not in ('CW', 'AM') or cfg['out_a'] not in VALID_OUTPUTS or cfg['out_b'] not in VALID_OUTPUTS:
        raise ValueError('WCFG mode or output selection is invalid.')
    return cfg

def handle_stm_line(line):
    if not line.upper().startswith('WCFG,'):
        return False
    try:
        cfg = parse_wcfg(line)
        print('Validated WCFG:', cfg)
        if APPLY_WCFG_TO_DAC:
            max5885 = MAX5885Signal(0x40001000)
            actual = max5885.configure_wireless(**cfg)
            uart_send('STATUS,PYNQ APPLIED %dMHz %s' % (round(cfg['carrier_hz'] / 1e6), cfg['mode']))
            uart_send('LOG,A=%s B=%s SD=%dmV' % (cfg['out_a'], cfg['out_b'], round(cfg['sd_vrms'] * 1000)))
            print('Applied:', actual)
        else:
            uart_send('STATUS,PYNQ WCFG VALID')
    except Exception as exc:
        uart_send('ERR,WCFG %s' % str(exc)[:80])
        raise
    return True

for line in received:
    handle_stm_line(line)


In [ ]:
# Cell 5: run while adjusting the STM32 screen. Interrupt this cell in Jupyter when finished.
print('Listening for STM32 WCFG commands; Jupyter Interrupt stops this cell.')
try:
    while True:
        line = uart_read_line(timeout_s=0.5)
        if line is not None:
            handle_stm_line(line)
except KeyboardInterrupt:
    print('UART listener stopped.')
